# Deutsch Algorithm — Teaching Notebook

The Deutsch algorithm is the smallest clean example showing how **superposition, phase kickback, and interference** can reveal a global property of a function with one oracle query.

This notebook is designed to be **self-explanatory for classroom teaching**. It moves from the problem statement and mathematics to quantum circuits, Qiskit implementation, interpretation, limitations, and exercises.

### Learning objectives
- distinguish constant from balanced one-bit Boolean functions
- derive the algorithm algebraically
- understand phase kickback
- implement all four possible Deutsch oracles
- compare classical and quantum query complexity

## 0. Installation
Run this only if Qiskit is not already installed.

In [1]:
%pip install -q qiskit qiskit-aer matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and helper functions

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, pi

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title='Measurement counts'):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8,4))
    plt.bar(keys, vals)
    plt.xlabel('bit string')
    plt.ylabel('counts')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

/var/folders/ch/1rsqd02j4xjgnwbds4jk8yfm0000gn/T/ipykernel_8570/2428700715.py:6: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile


# Part I — The problem
We are given a black-box Boolean function

$$f:\{0,1\}\rightarrow\{0,1\}.$$

There are four possible functions:

| $f(0)$ | $f(1)$ | type |
|---:|---:|---|
|0|0|constant|
|1|1|constant|
|0|1|balanced|
|1|0|balanced|

The task is **not** to learn both values. The task is only to determine whether the function is constant or balanced.

## 2. Classical query complexity
In the deterministic classical setting, one function call is insufficient: after observing only $f(0)$, we still do not know $f(1)$. In the worst case we need **two queries**.

In [3]:
functions = {
    'constant 0': (0,0),
    'constant 1': (1,1),
    'balanced identity': (0,1),
    'balanced NOT': (1,0),
}
functions

{'constant 0': (0, 0),
 'constant 1': (1, 1),
 'balanced identity': (0, 1),
 'balanced NOT': (1, 0)}

# Part II — Quantum oracle
The reversible oracle is

$$U_f|x,y\rangle = |x, y\oplus f(x)\rangle.$$

A quantum oracle must be unitary, so instead of replacing $y$ by $f(x)$, it XORs $f(x)$ into an ancilla.

## 3. Why the ancilla starts in $|1\rangle$
We prepare

$$|0\rangle|1\rangle.$$

After Hadamards,

$$|+\rangle|-\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}\otimes\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

The state $|-\rangle$ is an eigenstate of $X$ with eigenvalue $-1$. Therefore XORing into this ancilla turns the function value into a **phase**:

$$U_f|x\rangle|-\rangle=(-1)^{f(x)}|x\rangle|-\rangle.$$

This is **phase kickback**.

## 4. Algebraic derivation
After the oracle, the first qubit is

$$\frac{(-1)^{f(0)}|0\rangle+(-1)^{f(1)}|1\rangle}{\sqrt2}.$$

Apply a final Hadamard:

- if $f(0)=f(1)$, the relative phase is the same and the result is $|0\rangle$;
- if $f(0)\neq f(1)$, the relative phase differs and the result is $|1\rangle$.

So **0 means constant and 1 means balanced**.

# Part III — Build the four oracles

In [4]:
def deutsch_oracle(kind):
    qc = QuantumCircuit(2, name=kind)
    if kind == 'constant0':
        pass
    elif kind == 'constant1':
        qc.x(1)
    elif kind == 'balanced_identity':
        qc.cx(0,1)
    elif kind == 'balanced_not':
        qc.x(1)
        qc.cx(0,1)
    else:
        raise ValueError('unknown oracle')
    return qc

for kind in ['constant0','constant1','balanced_identity','balanced_not']:
    print(kind)
    print(deutsch_oracle(kind).draw())

constant0
     
q_0: 
     
q_1: 
     
constant1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘
balanced_identity
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘
balanced_not
               
q_0: ───────■──
     ┌───┐┌─┴─┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘


## 5. Full Deutsch circuit

In [5]:
def deutsch_circuit(kind):
    qc = QuantumCircuit(2,1)
    qc.x(1)
    qc.h([0,1])
    qc.compose(deutsch_oracle(kind), inplace=True)
    qc.h(0)
    qc.measure(0,0)
    return qc

qc = deutsch_circuit('balanced_identity')
print(qc.draw())

     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 


## 6. Run all four functions

In [6]:
for kind in ['constant0','constant1','balanced_identity','balanced_not']:
    counts = run_counts(deutsch_circuit(kind))
    print(f'{kind:20s} -> {counts}')

constant0            -> {'0': 2048}
constant1            -> {'0': 2048}
balanced_identity    -> {'1': 2048}
balanced_not         -> {'1': 2048}


## 7. Expected interpretation
Ignoring hardware noise, the measurement is deterministic:

$$0\Rightarrow\text{constant},\qquad 1\Rightarrow\text{balanced}.$$

The important quantum resource is not simply parallel evaluation. The oracle writes function information into **relative phase**, and the final Hadamard converts that phase difference into a measurable bit.

# Part IV — Statevector view

In [7]:
def state_before_final_h(kind):
    qc = QuantumCircuit(2)
    qc.x(1); qc.h([0,1])
    qc.compose(deutsch_oracle(kind), inplace=True)
    return Statevector.from_instruction(qc)

for kind in ['constant0','balanced_identity']:
    print(kind, np.round(state_before_final_h(kind).data,3))

constant0 [ 0.5+0.j  0.5+0.j -0.5+0.j -0.5+0.j]
balanced_identity [ 0.5+0.j -0.5+0.j -0.5+0.j  0.5+0.j]


# Part V — Classical versus quantum
| Feature | Classical deterministic | Deutsch |
|---|---:|---:|
|Input size|1 bit|1 qubit input|
|Worst-case oracle queries|2|1|
|Output|constant/balanced|constant/balanced|
|Key mechanism|direct evaluation|phase kickback + interference|

Deutsch's algorithm is mainly pedagogical. Its generalization, Deutsch–Jozsa, exposes the same principle on $n$ input bits.

# Part VI — Common misconceptions
1. **The quantum computer does not read both $f(0)$ and $f(1)$ individually.** It extracts one global property.
2. **Superposition alone is not enough.** Interference is required to turn phase information into a deterministic answer.
3. **The ancilla is not measured.** It enables phase kickback.
4. **The oracle must be reversible/unitary.**

# Part VII — Exercises
1. Write the state after every gate for the balanced identity oracle.
2. Show explicitly that $X|-\rangle=-|-\rangle$.
3. Build the balanced-NOT oracle using only $X$ and CNOT.
4. What happens if the ancilla starts in $|0\rangle$ rather than $|1\rangle$?
5. Explain why one classical deterministic query is insufficient.

# Part VIII — Solutions
**1.** Start $|01\rangle$, then $|+\rangle|-\rangle$, then oracle creates opposite phases on the first qubit, then the final $H$ maps the first qubit to $|1\rangle$.

**2.** $|-\rangle=(|0\rangle-|1\rangle)/\sqrt2$. Applying $X$ swaps $|0\rangle$ and $|1\rangle$, giving $(|1\rangle-|0\rangle)/\sqrt2=-|-\rangle$.

**4.** After $H$, the ancilla becomes $|+\rangle$, which is an $X$ eigenstate with eigenvalue $+1$; the desired function-dependent phase disappears.

**5.** One observed value leaves two possible function classes consistent with the observation.

# Compact reference
$$|0\rangle|1\rangle \xrightarrow{H\otimes H}\frac{|0\rangle+|1\rangle}{\sqrt2}|-\rangle\xrightarrow{U_f}\frac{(-1)^{f(0)}|0\rangle+(-1)^{f(1)}|1\rangle}{\sqrt2}|-\rangle\xrightarrow{H}\begin{cases}|0\rangle,&f(0)=f(1)\\|1\rangle,&f(0)\ne f(1).\end{cases}$$